# QAT MNISTモデル

## モデルの構造

MNIST手書き数字認識用の畳み込みニューラルネットワーク（CNN）で、量子化対応（QAT: Quantization Aware Training）を実装。

- **入力**: 1チャネル、28x28ピクセルのグレースケール画像
- **畳み込み層1**: Conv2d(1→4, カーネル3x3, パディングなし) → MaxPool(2x2)
- **畳み込み層2**: Conv2d(4→8, カーネル3x3, パディングなし) → MaxPool(2x2)
- **全結合層**: 200→10（クラス数）
- 各畳み込み層・全結合層に対し、重み・活性化の量子化（LSQ, PACT）を適用

## アキュラシー

最終的なテスト精度は**約98%**前後。

## 学習内容

- **データセット**: torchvisionのMNIST（train/testセット）
- **前処理**: ToTensor, Normalize(mean=0.1307, std=0.3081)
- **学習手順**:
  - 最初の数エポックは量子化なしでウォームアップ
  - 以降は重み量子化を有効化
  - 最適化: Adam（量子化パラメータはweight decayなし）
  - 損失関数: CrossEntropyLoss
- **エクスポート**: 学習済みモデルを量子化済みパラメータ（npz/json）として保存し、C言語ヘッダ生成や組み込み推論に利用可能


In [ ]:
# qat_mnist_end2end.py
import os
import math
import json
import argparse
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# =========================================================
# Utils
# =========================================================
def _calc_qrange(bit: int, is_weight: bool):
    if is_weight:
        qn = -(2 ** (bit - 1))
        qp = (2 ** (bit - 1)) - 1
    else:
        qn = 0
        qp = (2 ** bit) - 1
    return qn, qp


def _grad_scale(x: torch.Tensor, scale: float):
    # forward: identity, backward: grad * scale
    return (x - x.detach()) * scale + x.detach()


def _round_ste(x: torch.Tensor):
    # forward: round, backward: identity
    return (x.round() - x).detach() + x


# =========================================================
# PACT (Parametric Clipping Activation)
# =========================================================
class PACTReLU(nn.Module):
    def __init__(self, num_channels: int, init_alpha: float = 6.0, per_channel: bool = True):
        super().__init__()
        self.per_channel = per_channel
        if per_channel:
            self.alpha = nn.Parameter(torch.full((num_channels, 1, 1), float(init_alpha)))
        else:
            self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # learnable upper bound (softplus for positivity)
        alpha_pos = F.softplus(self.alpha)
        y = torch.clamp(x, min=0.0)
        y = torch.minimum(y, alpha_pos)
        return y, alpha_pos


# =========================================================
# LSQ (Learned Step Size Quantization)
# =========================================================
class LSQQuantizer(nn.Module):
    """
    - Weights: per-channel scale (out_channels)
    - Activations: per-tensor scale
    Returns: (q_int, scale(broadcastable), dequantized)
    """
    def __init__(
        self,
        out_channels: int,
        wt_shape: Optional[torch.Size],
        is_weight: bool,
        bit: int = 8,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.is_weight = is_weight
        self.bit = bit
        self.eps = eps

        qn, qp = _calc_qrange(bit, is_weight)
        self.register_buffer("qn", torch.tensor(float(qn)))
        self.register_buffer("qp", torch.tensor(float(qp)))

        if is_weight:
            self.scale = nn.Parameter(torch.ones(out_channels))
            self._inited = False
        else:
            self.scale = nn.Parameter(torch.tensor(1.0))
            self._inited = False

    @torch.no_grad()
    def _init_scale(self, x: torch.Tensor):
        qp = self.qp.item()
        if self.is_weight:
            c = x.shape[0]
            mean_abs = x.view(c, -1).abs().mean(dim=1) + self.eps
            s = 2.0 * mean_abs / math.sqrt(qp)
            self.scale.data.copy_(s)
        else:
            mean_abs = x.abs().mean() + self.eps
            s = 2.0 * mean_abs / math.sqrt(qp)
            self.scale.data.copy_(s)
        self._inited = True

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if not self._inited:
            self._init_scale(x.detach())

        qn, qp = self.qn, self.qp

        if self.is_weight:
            s = torch.abs(self.scale)
            if x.ndim == 4:
                s_b = s.view(-1, 1, 1, 1)
                n_perchannel = x[0].numel()
            elif x.ndim == 2:
                s_b = s.view(-1, 1)
                n_perchannel = x[0].numel()
            else:
                s_b = s.view(s.shape[0], *([1] * (x.ndim - 1)))
                n_perchannel = x[0].numel()

            grad_factor = 1.0 / math.sqrt(n_perchannel * qp.item())
            s_b = _grad_scale(s_b, grad_factor)

            x_div_s = x / (s_b + self.eps)
            x_bar = torch.clamp(x_div_s, min=qn.item(), max=qp.item())
            q = _round_ste(x_bar)
            x_hat = q * s_b
            return q, s_b, x_hat
        else:
            s = torch.abs(self.scale)
            N = x.numel()
            grad_factor = 1.0 / math.sqrt(max(N, 1) * qp.item())
            s = _grad_scale(s, grad_factor)

            x_div_s = x / (s + self.eps)
            x_bar = torch.clamp(x_div_s, min=qn.item(), max=qp.item())
            q = _round_ste(x_bar)
            x_hat = q * s
            return q, s, x_hat


# =========================================================
# QAT Layers
# =========================================================
class QATConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, stride, padding, bit_w=8, bit_a=8):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, stride, padding, bias=True)
        self.lsq = LSQQuantizer(out_ch, self.conv.weight.shape, is_weight=True, bit=bit_w)
        self.pact = PACTReLU(out_ch)
        self.act_lsq = LSQQuantizer(out_ch, None, is_weight=False, bit=bit_a)
        self.use_weight_quant = True  # warmup用

    def forward(self, x):
        if self.use_weight_quant:
            _, _, w_deq = self.lsq(self.conv.weight)
        else:
            w_deq = self.conv.weight

        x = F.conv2d(x, w_deq, self.conv.bias, self.conv.stride, self.conv.padding)
        x, alpha = self.pact(x)
        _, act_scale, x_deq = self.act_lsq(x)
        return x_deq, None, None, self.conv.bias, act_scale, alpha


class QATLinear(nn.Module):
    def __init__(self, in_ch, out_ch, bit_w=8):
        super().__init__()
        self.fc = nn.Linear(in_ch, out_ch, bias=True)
        self.lsq = LSQQuantizer(out_ch, self.fc.weight.shape, is_weight=True, bit=bit_w)
        self.use_weight_quant = True

    def forward(self, x):
        if self.use_weight_quant:
            _, _, w_deq = self.lsq(self.fc.weight)
        else:
            w_deq = self.fc.weight
        x = F.linear(x, w_deq, self.fc.bias)
        return x, None, None, self.fc.bias


# =========================================================
# Network
# =========================================================
class QATNet(nn.Module):
    """
    MNIST (N,1,28,28)
    Conv(1->4,k3,p0)->26x26->Pool->13x13
    Conv(4->8,k3,p0)->11x11->Pool->5x5
    Flatten -> 8*5*5=200 -> FC(200->10)
    """
    def __init__(self, bit_w=8, bit_a=8):
        super().__init__()
        self.conv1 = QATConv2d(1, 4, 3, 1, 0, bit_w=bit_w, bit_a=bit_a)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = QATConv2d(4, 8, 3, 1, 0, bit_w=bit_w, bit_a=bit_a)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc = QATLinear(200, 10, bit_w=bit_w)

    def forward(self, x):
        x, *_ = self.conv1(x)
        x = self.pool1(x)
        x, *_ = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x, *_ = self.fc(x)
        return x


# =========================================================
# Data
# =========================================================
def get_loaders(batch_size=128, num_workers=2):
    transform = transforms.Compose([
        transforms.ToTensor(),                       # to [0,1]
        transforms.Normalize((0.1307,), (0.3081,)),  # standard MNIST normalize
    ])
    train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader


# =========================================================
# Optimizer (param group: scale/alphaはWDなし)
# =========================================================
def make_optimizer(model, base_lr=1e-3, wd=1e-4):
    scale_params, alpha_params, other_params = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "scale" in n:
            scale_params.append(p)
        elif "alpha" in n:
            alpha_params.append(p)
        else:
            other_params.append(p)

    return torch.optim.Adam([
        {"params": other_params, "lr": base_lr, "weight_decay": wd},
        {"params": scale_params, "lr": base_lr, "weight_decay": 0.0},
        {"params": alpha_params, "lr": base_lr, "weight_decay": 0.0},
    ])


# =========================================================
# Train / Test
# =========================================================
def set_weight_quant(model: QATNet, enabled: bool):
    model.conv1.use_weight_quant = enabled
    model.conv2.use_weight_quant = enabled
    model.fc.use_weight_quant = enabled


def train_one_epoch(model, device, loader, optimizer, criterion, epoch, log_interval=100):
    model.train()
    running = 0.0
    for i, (x, y) in enumerate(loader, 1):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running += loss.item()
        if i % log_interval == 0:
            print(f"Epoch [{epoch}] Batch [{i}/{len(loader)}] Loss: {running / i:.4f}")


@torch.no_grad()
def test(model, device, loader, criterion):
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        out = model(x)
        loss_sum += criterion(out, y).item()
        pred = out.argmax(dim=1)
        total += y.size(0)
        correct += (pred == y).sum().item()
    acc = 100.0 * correct / total
    print(f"Test Loss: {loss_sum / len(loader):.4f}, Accuracy: {acc:.2f}%")
    return acc


# =========================================================
# Export helpers (fast, vectorized)
# =========================================================
def quantize_multiplier_array(m: np.ndarray):
    """
    m: np.ndarray of float (>=0)
    return: (mult:uint32 array, rshift:int32 array)  s.t.  m ≈ mult / 2^rshift
    impl: m = sig * 2^exp (0.5<=sig<1), mult=round(sig*2^31), rshift=31-exp
    """
    m = np.asarray(m, dtype=np.float64)
    mult = np.zeros_like(m, dtype=np.int64)
    rshift = np.zeros_like(m, dtype=np.int32)

    valid = np.isfinite(m) & (m > 0.0)
    if not np.any(valid):
        return mult.astype(np.uint32), rshift

    sig, exp = np.frexp(m[valid])                         # m = sig * 2**exp
    mult_valid = np.round(sig * (1 << 31)).astype(np.int64)
    overflow = (mult_valid == (1 << 31))
    if np.any(overflow):
        mult_valid[overflow] >>= 1
        exp[overflow] += 1

    rshift_valid = (31 - exp).astype(np.int32)
    rshift_valid = np.maximum(rshift_valid, 0)

    mult[valid] = mult_valid
    rshift[valid] = rshift_valid
    return mult.astype(np.uint32), rshift


def clamp_round_int8(x):
    return np.clip(np.round(x), -128, 127).astype(np.int8)


def fold_input_normalize_into_conv1(conv, mean=0.1307, std=0.3081):
    # conv.weight: (Cout, Cin=1, Kh, Kw)
    W = conv.weight.detach().clone()
    b = conv.bias.detach().clone() if conv.bias is not None else torch.zeros(W.size(0), device=W.device)

    Wp = W / std
    kernel_sum = Wp.sum(dim=(1, 2, 3))  # (Cout,)
    bp = b - (mean) * kernel_sum
    return Wp, bp


def export_qat_conv(layer, s_x, s_out=None, first_conv_fold_normalize=False):
    conv = layer.conv
    if first_conv_fold_normalize:
        Wf, bf = fold_input_normalize_into_conv1(conv)
    else:
        Wf = conv.weight.detach().clone()
        bf = conv.bias.detach().clone() if conv.bias is not None else torch.zeros(conv.weight.size(0), device=Wf.device)

    s_w = np.abs(layer.lsq.scale.detach().clone().cpu().numpy())  # (Cout,)
    if s_out is None:
        s_out = float(layer.act_lsq.scale.detach().cpu().numpy())

    Wn = Wf.cpu().numpy()
    Cout = Wn.shape[0]
    w_int8 = np.zeros_like(Wn, dtype=np.int8)
    for c in range(Cout):
        w_int8[c] = clamp_round_int8(Wn[c] / (s_w[c] + 1e-8))

    m = (s_x * s_w) / (s_out + 1e-12)                     # shape: (Cout,)
    mult, rshift = quantize_multiplier_array(m)
    b_int32 = np.round(bf.cpu().numpy() / (s_out + 1e-12)).astype(np.int32)

    return {
        "weight_int8": w_int8,
        "bias_int32": b_int32,
        "w_scale": s_w.astype(np.float32),
        "in_scale": np.array([s_x], np.float32),
        "out_scale": np.array([s_out], np.float32),
        "requant_mult": mult,
        "requant_rshift": rshift,
        "output_unsigned": True,  # ReLU後はu8運用を想定
    }


def export_qat_linear(layer, s_x, s_out=None):
    fc = layer.fc
    Wf = fc.weight.detach().clone()
    bf = fc.bias.detach().clone() if fc.bias is not None else torch.zeros(fc.weight.size(0), device=Wf.device)

    s_w = np.abs(layer.lsq.scale.detach().clone().cpu().numpy())  # (Cout,)
    if s_out is None:
        s_out = 1.0 / 128.0

    Wn = Wf.cpu().numpy()
    Cout = Wn.shape[0]
    w_int8 = np.zeros_like(Wn, dtype=np.int8)
    for c in range(Cout):
        w_int8[c] = clamp_round_int8(Wn[c] / (s_w[c] + 1e-8))

    m = (s_x * s_w) / (s_out + 1e-12)
    mult, rshift = quantize_multiplier_array(m)
    b_int32 = np.round(bf.cpu().numpy() / (s_out + 1e-12)).astype(np.int32)

    return {
        "weight_int8": w_int8,
        "bias_int32": b_int32,
        "w_scale": s_w.astype(np.float32),
        "in_scale": np.array([s_x], np.float32),
        "out_scale": np.array([s_out], np.float32),
        "requant_mult": mult,
        "requant_rshift": rshift,
        "output_unsigned": False,
    }


def export_qatnet_to_int8_params(model, fname_npz="qat_export.npz",
                                 fold_input_normalize=True,
                                 input_scale=None, input_signed=True):
    """
    model: trained QATNet
    fold_input_normalize: True -> fold Normalize(mean=0.1307,std=0.3081) into conv1
    input_scale: first input scale s_x (signed int8 => default 1/128)
    input_signed: whether input is signed int8 with zero_point=0
    """
    model.eval()

    s_in = float(1.0 / 128.0 if input_scale is None else input_scale)

    # conv1
    conv1_dict = export_qat_conv(
        model.conv1,
        s_x=s_in,
        s_out=float(model.conv1.act_lsq.scale.detach().cpu().numpy()),
        first_conv_fold_normalize=fold_input_normalize
    )
    # conv2
    conv2_dict = export_qat_conv(
        model.conv2,
        s_x=float(conv1_dict["out_scale"][0]),
        s_out=float(model.conv2.act_lsq.scale.detach().cpu().numpy()),
        first_conv_fold_normalize=False
    )
    # fc
    fc_dict = export_qat_linear(
        model.fc,
        s_x=float(conv2_dict["out_scale"][0]),
        s_out=1.0/128.0  # 最終出力は対称int8で受け渡し
    )

    export = {
        "input": {
            "signed_int8": bool(input_signed),
            "scale": np.array([s_in], np.float32),
            "zero_point": np.array([0], np.int32)
        },
        "conv1": conv1_dict,
        "conv2": conv2_dict,
        "fc": fc_dict,
    }

    np.savez_compressed(
        fname_npz,
        **{
            "input_scale": export["input"]["scale"],
            "conv1_weight_int8": export["conv1"]["weight_int8"],
            "conv1_bias_int32": export["conv1"]["bias_int32"],
            "conv1_w_scale": export["conv1"]["w_scale"],
            "conv1_in_scale": export["conv1"]["in_scale"],
            "conv1_out_scale": export["conv1"]["out_scale"],
            "conv1_requant_mult": export["conv1"]["requant_mult"],
            "conv1_requant_rshift": export["conv1"]["requant_rshift"],

            "conv2_weight_int8": export["conv2"]["weight_int8"],
            "conv2_bias_int32": export["conv2"]["bias_int32"],
            "conv2_w_scale": export["conv2"]["w_scale"],
            "conv2_in_scale": export["conv2"]["in_scale"],
            "conv2_out_scale": export["conv2"]["out_scale"],
            "conv2_requant_mult": export["conv2"]["requant_mult"],
            "conv2_requant_rshift": export["conv2"]["requant_rshift"],

            "fc_weight_int8": export["fc"]["weight_int8"],
            "fc_bias_int32": export["fc"]["bias_int32"],
            "fc_w_scale": export["fc"]["w_scale"],
            "fc_in_scale": export["fc"]["in_scale"],
            "fc_out_scale": export["fc"]["out_scale"],
            "fc_requant_mult": export["fc"]["requant_mult"],
            "fc_requant_rshift": export["fc"]["requant_rshift"],
        }
    )

    with open(os.path.splitext(fname_npz)[0] + ".json", "w") as f:
        json.dump({
            "input": {
                "signed_int8": export["input"]["signed_int8"],
                "scale": export["input"]["scale"].tolist(),
                "zero_point": [0],
            },
            "conv1": {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                      for k, v in export["conv1"].items() if k not in ("weight_int8", "bias_int32")},
            "conv2": {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                      for k, v in export["conv2"].items() if k not in ("weight_int8", "bias_int32")},
            "fc":    {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                      for k, v in export["fc"].items() if k not in ("weight_int8", "bias_int32")},
        }, f, indent=2)

    return export


# =========================================================
# Main
# =========================================================
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--wd", type=float, default=1e-4)
    parser.add_argument("--warmup-epochs", type=int, default=1)
    parser.add_argument("--num-workers", type=int, default=2)
    parser.add_argument("--skip-train", action="store_true")
    parser.add_argument("--ckpt", type=str, default="qat_mnist.pth")
    parser.add_argument("--export-npz", type=str, default="qat_export.npz")
    parser.add_argument("--no-fold-normalize", action="store_true",
                        help="conv1へのNormalize折り込みを無効化（推論入力を int8(-128..127)で与える場合は True 推奨）")
    parser.add_argument("--input-scale", type=float, default=1/128,
                        help="推論入力のスケール(signed int8想定). 通常 1/128")
    args = parser.parse_known_args()[0]

    torch.manual_seed(0)
    np.random.seed(0)
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Data
    train_loader, test_loader = get_loaders(batch_size=args.batch_size, num_workers=args.num_workers)

    # Model / Opt
    model = QATNet(bit_w=8, bit_a=8).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, base_lr=args.lr, wd=args.wd)

    # Train (optional)
    if not args.skip_train:
        for epoch in range(1, args.epochs + 1):
            set_weight_quant(model, enabled=(epoch > args.warmup_epochs))
            train_one_epoch(model, device, train_loader, optimizer, criterion, epoch)
            test(model, device, test_loader, criterion)
        torch.save(model.state_dict(), args.ckpt)
        print(f"Saved model to {args.ckpt}")
    else:
        if os.path.exists(args.ckpt):
            model.load_state_dict(torch.load(args.ckpt, map_location=device))
            print(f"Loaded checkpoint from {args.ckpt}")
        else:
            raise FileNotFoundError(f"--skip-train 指定ですが ckpt が存在しません: {args.ckpt}")

    # Export
    export = export_qatnet_to_int8_params(
        model,
        fname_npz=args.export_npz,
        fold_input_normalize=not args.no_fold_normalize,  # ← 絶対に実行時に --no-fold-normalize を付けてください
        input_scale=args.input_scale,
        input_signed=True
    )
    print(f"Exported to {args.export_npz} and {os.path.splitext(args.export_npz)[0] + '.json'}")


if __name__ == "__main__":
    main()


Epoch [1] Batch [100/469] Loss: 1.4056
Epoch [1] Batch [200/469] Loss: 0.8864
Epoch [1] Batch [300/469] Loss: 0.6844
Epoch [1] Batch [400/469] Loss: 0.5666
Test Loss: 0.1654, Accuracy: 95.35%
Epoch [2] Batch [100/469] Loss: 0.1797
Epoch [2] Batch [200/469] Loss: 0.1678
Epoch [2] Batch [300/469] Loss: 0.1608
Epoch [2] Batch [400/469] Loss: 0.1564
Test Loss: 0.1115, Accuracy: 96.65%
Epoch [3] Batch [100/469] Loss: 0.1273
Epoch [3] Batch [200/469] Loss: 0.1260
Epoch [3] Batch [300/469] Loss: 0.1210
Epoch [3] Batch [400/469] Loss: 0.1196
Test Loss: 0.0950, Accuracy: 97.12%
Epoch [4] Batch [100/469] Loss: 0.1010
Epoch [4] Batch [200/469] Loss: 0.1051
Epoch [4] Batch [300/469] Loss: 0.1036
Epoch [4] Batch [400/469] Loss: 0.1002
Test Loss: 0.0817, Accuracy: 97.54%
Epoch [5] Batch [100/469] Loss: 0.0953
Epoch [5] Batch [200/469] Loss: 0.0912
Epoch [5] Batch [300/469] Loss: 0.0898
Epoch [5] Batch [400/469] Loss: 0.0895
Test Loss: 0.0738, Accuracy: 97.69%
Epoch [6] Batch [100/469] Loss: 0.0823
E

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Exported npz (qat_export.npz など) から C ヘッダ(qcnn_params.h)を生成するスクリプト

使い方(スクリプト):
  python make_qheader.py --npz qat_export.npz --out qcnn_params.h --prefix Q --dump-csv
"""
import os
import sys
import numpy as np
from pathlib import Path
import argparse

# =========================
# ユーザ設定（デフォルト）
# =========================
NPZ_PATH_DEFAULT   = "D:/cfs/final/train/qat_export.npz"  # 実ファイルへ
OUT_HEADER_DEFAULT = "qcnn_params.h"
PREFIX_DEFAULT     = "Q"
DUMP_CSV_DEFAULT   = True

# ---------- ユーティリティ ----------
def _format_1d_lines(arr: np.ndarray, per_line: int, level: int) -> str:
    is_float = np.issubdtype(arr.dtype, np.floating)
    if is_float:
        vals = [f"{float(x):.10g}" for x in arr.tolist()]
    else:
        vals = [str(int(x)) for x in arr.astype(np.int64).tolist()]
    chunks = [", ".join(vals[i:i+per_line]) for i in range(0, len(vals), per_line)]
    indent = "  " * (level + 1)
    return "{ " + (",\n" + indent).join(chunks) + " }"

def c_array(dtype, name, data, per_line=16, const=True, static=True):
    data = np.asarray(data)
    quals = []
    if static: quals.append("static")
    if const:  quals.append("const")
    qual = " ".join(quals)
    shape = "".join(f"[{d}]" for d in data.shape)
    header = f"{qual} {dtype} {name}{shape} = "

    def format_nd(arr, level=0):
        if arr.ndim == 1:
            return _format_1d_lines(arr, per_line=per_line, level=level)
        else:
            inner = ",\n".join(("  "*(level+1)) + format_nd(sub, level+1) for sub in arr)
            return "{\n" + inner + "\n" + ("  "*level) + "}"

    body = format_nd(data)
    return header + body + ";\n\n"

def c_array_float(name, data, const=True, static=True, per_line=8):
    return c_array("float", name, np.asarray(data, dtype=np.float32), per_line=per_line, const=const, static=static)

def c_array_u32(name, data, const=True, static=True):
    return c_array("uint32_t", name, np.asarray(data, dtype=np.uint32), const=const, static=static)

def c_array_i32(name, data, const=True, static=True):
    return c_array("int32_t", name, np.asarray(data, dtype=np.int32), const=const, static=static)

def c_array_i8(name, data, const=True, static=True):
    return c_array("int8_t", name, np.asarray(data, dtype=np.int8), const=const, static=static)

def c_array_u8(name, data, const=True, static=True):
    return c_array("uint8_t", name, np.asarray(data, dtype=np.uint8), const=const, static=static)

def _shape_summary(Z):
    return {
        "conv1_weight": Z["conv1_weight_int8"].shape,
        "conv1_bias":   Z["conv1_bias_int32"].shape,
        "conv2_weight": Z["conv2_weight_int8"].shape,
        "conv2_bias":   Z["conv2_bias_int32"].shape,
        "fc_weight":    Z["fc_weight_int8"].shape,
        "fc_bias":      Z["fc_bias_int32"].shape,
    }

def dump_csvs(Z, out_dir="dump_csv"):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    def save(name, key):
        A = Z[key]
        if A.ndim == 1:
            A2 = A.reshape(1, -1)
        else:
            A2 = A.reshape(A.shape[0], -1)
        fmt = "%d" if np.issubdtype(A.dtype, np.integer) else "%.10g"
        np.savetxt(out / f"{name}.csv", A2, delimiter=",", fmt=fmt)
    save("conv1_weight_int8", "conv1_weight_int8")
    save("conv1_bias_int32",  "conv1_bias_int32")
    save("conv2_weight_int8", "conv2_weight_int8")
    save("conv2_bias_int32",  "conv2_bias_int32")
    save("fc_weight_int8",    "fc_weight_int8")
    save("fc_bias_int32",     "fc_bias_int32")
    print(f"Saved CSVs to ./{out_dir}")

# ---------- メイン生成 ----------
def generate_header(npz_path: str,
                    out_header: str = OUT_HEADER_DEFAULT,
                    prefix: str = PREFIX_DEFAULT,
                    dump_csv: bool = DUMP_CSV_DEFAULT):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ が見つかりません: {npz_path}")

    Z = np.load(npz_path)

    required = [
        "input_scale",
        "conv1_weight_int8","conv1_bias_int32","conv1_w_scale","conv1_in_scale","conv1_out_scale","conv1_requant_mult","conv1_requant_rshift",
        "conv2_weight_int8","conv2_bias_int32","conv2_w_scale","conv2_in_scale","conv2_out_scale","conv2_requant_mult","conv2_requant_rshift",
        "fc_weight_int8","fc_bias_int32","fc_w_scale","fc_in_scale","fc_out_scale","fc_requant_mult","fc_requant_rshift",
    ]
    missing = [k for k in required if k not in Z]
    if missing:
        raise KeyError(f"NPZ に不足しているキー: {missing}")

    # 入力
    input_scale = Z["input_scale"].astype(np.float32)

    # conv1
    c1_w   = Z["conv1_weight_int8"].astype(np.int8)
    c1_b   = Z["conv1_bias_int32"].astype(np.int32)
    c1_ws  = Z["conv1_w_scale"].astype(np.float32)
    c1_is  = Z["conv1_in_scale"].astype(np.float32)
    c1_os  = Z["conv1_out_scale"].astype(np.float32)
    c1_mul = Z["conv1_requant_mult"].astype(np.uint32)
    c1_rsh = Z["conv1_requant_rshift"].astype(np.uint32)

    # conv2
    c2_w   = Z["conv2_weight_int8"].astype(np.int8)
    c2_b   = Z["conv2_bias_int32"].astype(np.int32)
    c2_ws  = Z["conv2_w_scale"].astype(np.float32)
    c2_is  = Z["conv2_in_scale"].astype(np.float32)
    c2_os  = Z["conv2_out_scale"].astype(np.float32)
    c2_mul = Z["conv2_requant_mult"].astype(np.uint32)
    c2_rsh = Z["conv2_requant_rshift"].astype(np.uint32)

    # fc
    fc_w   = Z["fc_weight_int8"].astype(np.int8)
    fc_b   = Z["fc_bias_int32"].astype(np.int32)
    fc_ws  = Z["fc_w_scale"].astype(np.float32)
    fc_is  = Z["fc_in_scale"].astype(np.float32)
    fc_os  = Z["fc_out_scale"].astype(np.float32)
    fc_mul = Z["fc_requant_mult"].astype(np.uint32)
    fc_rsh = Z["fc_requant_rshift"].astype(np.uint32)

    # 形状
    IN_C, IN_H, IN_W = 1, 28, 28
    C1_IC, C1_OC, C1_K = c1_w.shape[1], c1_w.shape[0], c1_w.shape[2]
    P1_H, P1_W = (IN_H - C1_K + 1)//2, (IN_W - C1_K + 1)//2
    C2_IC, C2_OC, C2_K = c2_w.shape[1], c2_w.shape[0], c2_w.shape[2]
    P2_H, P2_W = (P1_H - C2_K + 1)//2, (P1_W - C2_K + 1)//2
    FC_IN, FC_OUT = fc_w.shape[1], fc_w.shape[0]

    out_path = Path(out_header)
    guard = (prefix.upper() + "_QCNN_PARAMS_H_").replace("__","_")
    with out_path.open("w", newline="\n", encoding="utf-8") as f:
        pf = prefix

        f.write("// Auto-generated by make_qheader.py\n")
        f.write("#ifndef " + guard + "\n#define " + guard + "\n\n")
        f.write("#include <stdint.h>\n\n")

        # shapes
        f.write("// ---- Network shapes ----\n")
        f.write(f"#define IN_C   {IN_C}\n#define IN_H   {IN_H}\n#define IN_W   {IN_W}\n\n")
        f.write(f"#define C1_IC  {C1_IC}\n#define C1_OC  {C1_OC}\n#define C1_K   {C1_K}\n")
        f.write(f"#define P1_H   {P1_H}\n#define P1_W   {P1_W}\n\n")
        f.write(f"#define C2_IC  {C2_IC}\n#define C2_OC  {C2_OC}\n#define C2_K   {C2_K}\n")
        f.write(f"#define P2_H   {P2_H}\n#define P2_W   {P2_W}\n\n")
        f.write(f"#define FC_IN  {FC_IN}\n#define FC_OUT {FC_OUT}\n\n")

        # input quant
        f.write("// ---- Input quant ----\n")
        f.write(c_array_float(pf + "input_scale", input_scale.reshape(1)))

        # conv1
        f.write("// ---- Conv1 ----\n")
        f.write(c_array_i8 (pf + "conv1_weight_int8", c1_w))
        f.write(c_array_i32(pf + "conv1_bias_int32",  c1_b))
        f.write(c_array_float(pf + "conv1_w_scale",   c1_ws))
        f.write(c_array_float(pf + "conv1_in_scale",  c1_is.reshape(1)))
        f.write(c_array_float(pf + "conv1_out_scale", c1_os.reshape(1)))
        f.write(c_array_u32(pf + "conv1_requant_mult",   c1_mul))
        f.write(c_array_u32(pf + "conv1_requant_rshift", c1_rsh))

        # conv2
        f.write("// ---- Conv2 ----\n")
        f.write(c_array_i8 (pf + "conv2_weight_int8", c2_w))
        f.write(c_array_i32(pf + "conv2_bias_int32",  c2_b))
        f.write(c_array_float(pf + "conv2_w_scale",   c2_ws))
        f.write(c_array_float(pf + "conv2_in_scale",  c2_is.reshape(1)))
        f.write(c_array_float(pf + "conv2_out_scale", c2_os.reshape(1)))
        f.write(c_array_u32(pf + "conv2_requant_mult",   c2_mul))
        f.write(c_array_u32(pf + "conv2_requant_rshift", c2_rsh))

        # fc
        f.write("// ---- FC ----\n")
        f.write(c_array_i8 (pf + "fc_weight_int8", fc_w))
        f.write(c_array_i32(pf + "fc_bias_int32",  fc_b))
        f.write(c_array_float(pf + "fc_w_scale",   fc_ws))
        f.write(c_array_float(pf + "fc_in_scale",  fc_is.reshape(1)))
        f.write(c_array_float(pf + "fc_out_scale", fc_os.reshape(1)))
        f.write(c_array_u32(pf + "fc_requant_mult",   fc_mul))
        f.write(c_array_u32(pf + "fc_requant_rshift", fc_rsh))

        # notes
        f.write("// Notes:\n")
        f.write("// - requant_mult は unsigned 32bit (固定小数点)。右シフトは論理シフト想定。\n")
        f.write("// - conv1/conv2 の出力は ReLU 後 (u8) 運用を想定。fc は signed。\n")
        f.write("// - 推論では (int64 acc * mult + 0.5*2^rshift) >> rshift で再量子化してください。\n\n")
        f.write("#endif // " + guard + "\n")

    # サマリ
    shapes = _shape_summary(Z)
    print(f"[input]  scale: {float(input_scale[0]):.8f}")
    print("[conv1]  W {}, b {}, w_scale {}, in/out_scale {} / {}, requant {} {}".format(
        shapes['conv1_weight'], shapes['conv1_bias'], c1_ws.shape, c1_is.shape, c1_os.shape, c1_mul.shape, c1_rsh.shape))
    print("[conv2]  W {}, b {}, w_scale {}, in/out_scale {} / {}, requant {} {}".format(
        shapes['conv2_weight'], shapes['conv2_bias'], c2_ws.shape, c2_is.shape, c2_os.shape, c2_mul.shape, c2_rsh.shape))
    print("[fc]     W {}, b {}, w_scale {}, in/out_scale {} / {}, requant {} {}".format(
        shapes['fc_weight'], shapes['fc_bias'], fc_ws.shape, fc_is.shape, fc_os.shape, fc_mul.shape, fc_rsh.shape))

    if dump_csv:
        dump_csvs(Z)

    print(f"Wrote header to: {out_header}")

# -------- エントリポイント --------
def _parse_args_or_defaults():
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("--npz", type=str, help=".npz のパス")
    parser.add_argument("--out", type=str, help="出力 .h", default=OUT_HEADER_DEFAULT)
    parser.add_argument("--prefix", type=str, help="シンボル接頭辞", default=PREFIX_DEFAULT)
    parser.add_argument("--dump-csv", action="store_true", help="CSV も保存")
    parser.add_argument("--no-dump-csv", action="store_true", help="CSV を保存しない")
    if len(sys.argv) > 1 and any(a.startswith("--") for a in sys.argv[1:]):
        args = parser.parse_args()
        npz_path = args.npz if args.npz is not None else NPZ_PATH_DEFAULT
        dump_csv = DUMP_CSV_DEFAULT
        if args.dump_csv:
            dump_csv = True
        if args.no_dump_csv:
            dump_csv = False
        return npz_path, args.out, args.prefix, dump_csv
    else:
        return NPZ_PATH_DEFAULT, OUT_HEADER_DEFAULT, PREFIX_DEFAULT, DUMP_CSV_DEFAULT

if __name__ == "__main__":
    if "ipykernel_launcher" in sys.argv[0]:
        npz_path   = NPZ_PATH_DEFAULT
        out_header = OUT_HEADER_DEFAULT
        prefix     = PREFIX_DEFAULT
        dump_csv   = DUMP_CSV_DEFAULT
    else:
        npz_path, out_header, prefix, dump_csv = _parse_args_or_defaults()

    generate_header(npz_path=npz_path, out_header=out_header,
                    prefix=prefix, dump_csv=dump_csv)


[input]  scale: 0.00781250
[conv1]  W (4, 1, 3, 3), b (4,), w_scale (4,), in/out_scale (1,) / (1,), requant (4,) (4,)
[conv2]  W (8, 4, 3, 3), b (8,), w_scale (8,), in/out_scale (1,) / (1,), requant (8,) (8,)
[fc]     W (10, 200), b (10,), w_scale (10,), in/out_scale (1,) / (1,), requant (10,) (10,)
Saved CSVs to ./dump_csv
Wrote header to: qcnn_params.h


In [11]:
#!/usr/bin/env python3
import numpy as np
import torch
from torchvision import datasets, transforms

# 出力ファイル名
OUT_H = "indata.h"
OUT_C = "indata.c"

# 定数
NUM_CLASSES = 10
IN_C, IN_H, IN_W = 1, 28, 28
MEAN, STD = 0.1307, 0.3081  # MNIST標準統計

# ---- MNISTからラベル0~9を順番に1枚ずつ取得 ----
transform = transforms.Compose([transforms.ToTensor()])
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

imgs = [None] * NUM_CLASSES
for img, lab in test_ds:
    lab = int(lab)
    if imgs[lab] is None:
        imgs[lab] = img.squeeze(0).numpy()
    if all(x is not None for x in imgs):
        break

imgs = np.stack(imgs)                       # (10,28,28)
labels = np.arange(NUM_CLASSES, dtype=np.uint8)

# ---- 標準化 -> int8量子化（-128..127）----
x_norm = (imgs - MEAN) / STD
x_q = np.clip(np.round(128 * x_norm), -128, 127).astype(np.int8)  # (10,28,28)
x_q4 = x_q[:, None, :, :]  # (10,1,28,28)

# ---- フォーマット関数 ----
def format_2d(arr2d):
    rows = []
    for r in arr2d:
        rows.append("{ " + ", ".join(str(int(v)) for v in r.tolist()) + " }")
    return "{\n      " + ",\n      ".join(rows) + "\n    }"

def format_4d(arr4d):
    blocks = []
    for n in range(arr4d.shape[0]):
        img_block = format_2d(arr4d[n, 0])
        blocks.append("  { " + img_block + " }")
    return "{\n" + ",\n".join(blocks) + "\n}"

# ---- indata.h ----
with open(OUT_H, "w", encoding="utf-8") as f:
    f.write("// indata.h (auto-generated)\n")
    f.write("#ifndef INDATA_H_\n#define INDATA_H_\n\n")
    f.write("#include <stdint.h>\n#include \"qcnn_params.h\"\n\n")
    f.write(f"#define DATA_NUM {NUM_CLASSES}\n\n")
    f.write("// shape: [DATA_NUM][1][28][28] (int8)\n")
    f.write("extern int8_t indata[DATA_NUM][IN_C][IN_H][IN_W];\n\n")
    f.write("// labels: 0..9\n")
    f.write("extern const uint8_t inlabels[DATA_NUM];\n\n")
    f.write("#endif /* INDATA_H_ */\n")

# ---- indata.c ----
with open(OUT_C, "w", encoding="utf-8") as f:
    f.write("// indata.c (auto-generated)\n")
    f.write("#include <stdint.h>\n#include \"qcnn_params.h\"\n#include \"indata.h\"\n\n")
    f.write(f"int8_t indata[{NUM_CLASSES}][1][{IN_H}][{IN_W}] = ")
    f.write(format_4d(x_q4))
    f.write(";\n\n")
    f.write(f"const uint8_t inlabels[{NUM_CLASSES}] = ")
    f.write("{ " + ", ".join(str(int(v)) for v in labels.tolist()) + " };\n")

print(f"Wrote {OUT_H} and {OUT_C}")


Wrote indata.h and indata.c


In [13]:
# dump_mnist_bin.py
import numpy as np
from pathlib import Path
from torchvision import datasets, transforms

# ★ここを「実在するローカルドライブ直下のディレクトリ」に変えてください
ROOT = Path(r"D:/cfs/final").resolve()

S_IN = 1/128.0  # ヘッダの Qinput_scale と一致

ROOT.mkdir(parents=True, exist_ok=True)
tx = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# torchvision は <ROOT>/MNIST/raw/ に作業ディレクトリを作ります
ds = datasets.MNIST(root=str(ROOT), train=False, download=True, transform=tx)

X = np.empty((len(ds), 1, 28, 28), np.int8)
Y = np.empty((len(ds),), np.uint8)

for i, (img, label) in enumerate(ds):
    x = img.numpy().astype(np.float32) / S_IN       # 量子化スケールで割る
    x = np.clip(np.rint(x), -128, 127).astype(np.int8)
    X[i] = x
    Y[i] = label

(X).tofile(ROOT / "mnist_test_i8.bin")
(Y).tofile(ROOT / "mnist_test_labels.bin")
print(f"saved {X.shape} to {ROOT/'mnist_test_i8.bin'} / {ROOT/'mnist_test_labels.bin'}")


100.0%
100.0%
100.0%
100.0%


saved (10000, 1, 28, 28) to D:\cfs\final\mnist_test_i8.bin / D:\cfs\final\mnist_test_labels.bin


# 10000 sampleのデータ作成

In [7]:
# Cell 1: make_qmnist_header (bin -> qmnist_data.h)
import numpy as np
from pathlib import Path
from typing import Optional

def emit_header(
    images_bin: str,
    labels_bin: str,
    out_path: str,
    sym_images: str = "test_images",
    sym_labels: str = "test_labels",
    in_c: int = 1,
    in_h: int = 28,
    in_w: int = 28,
    limit: Optional[int] = None,
):
    """
    int8画像bin と uint8ラベルbin から Cヘッダ(qmnist_data.h)を生成。
    出力:
      static const int8_t  <sym_images>[NTEST][in_c][in_h][in_w];
      static const uint8_t <sym_labels>[NTEST];
    """
    images_bin = Path(images_bin)
    labels_bin = Path(labels_bin)
    out_path   = Path(out_path)

    # 読み込み
    X = np.fromfile(images_bin, dtype=np.int8)
    Y = np.fromfile(labels_bin, dtype=np.uint8)

    pixels_per = in_c * in_h * in_w
    if X.size % pixels_per != 0:
        print(f"[warn] images_bin size {X.size} is not multiple of {pixels_per}; truncating")
    n_total = X.size // pixels_per
    if n_total == 0:
        raise ValueError("images bin が空か、(C*H*W)の倍数になっていません。")

    if Y.size < n_total:
        n_total = Y.size  # ラベル数に合わせて短縮

    n = min(limit, n_total) if limit is not None else n_total

    X = X[: n * pixels_per].reshape(n, in_c, in_h, in_w)
    Y = Y[: n]

    # ヘッダ出力
    with out_path.open("w", encoding="utf-8") as f:
        f.write("// generated by notebook: make_qmnist_header\n")
        f.write("#pragma once\n#include <stdint.h>\n\n")
        f.write(f"// dataset size: {n}\n")
        f.write(f"#ifndef NTEST\n#define NTEST {n}\n#endif\n")
        f.write(f"#define IN_C {in_c}\n#define IN_H {in_h}\n#define IN_W {in_w}\n\n")

        # images
        f.write(f"static const int8_t {sym_images}[NTEST][IN_C][IN_H][IN_W] = {{\n")
        for i in range(n):
            f.write("  {\n")  # [C]
            for c in range(in_c):
                f.write("    {\n")  # [H]
                for h in range(in_h):
                    row = ", ".join(str(int(v)) for v in X[i, c, h])
                    f.write(f"      {{ {row} }}")
                    f.write(",\n" if h != in_h - 1 else "\n")
                f.write("    }")
                f.write(",\n" if c != in_c - 1 else "\n")
            f.write("  }")
            f.write(",\n" if i != n - 1 else "\n")
        f.write("};\n\n")

        # labels
        f.write(f"static const uint8_t {sym_labels}[NTEST] = {{\n  ")
        for i in range(n):
            f.write(str(int(Y[i])))
            if i != n - 1:
                f.write(", " if (i + 1) % 32 else ",\n  ")
        f.write("\n};\n")

    print(f"[ok] Wrote header: {out_path}  (NTEST default={n})")


In [8]:
# Cell 2: 使い方（パスを自分の環境に合わせて調整して実行）
from pathlib import Path

# ここだけ直せばOK
ROOT    = Path(r"D:/cfs/final").resolve()
SRC_DIR = ROOT / "src_c2"
IM_BIN  = ROOT / "mnist_test_i8.bin"
LB_BIN  = ROOT / "mnist_test_labels.bin"
OUT_H   = SRC_DIR / "qmnist_data.h"

# ヘッダ生成（NTEST=10000で切り出し）
emit_header(
    images_bin=str(IM_BIN),
    labels_bin=str(LB_BIN),
    out_path=str(OUT_H),
    sym_images="test_images",
    sym_labels="test_labels",
    in_c=1, in_h=28, in_w=28,
    limit=10000,
)

# PowerShell 用コンパイル一行コマンド（コピーして SRC_DIR で実行）
ps_one_liner = (
    'gcc -O3 -march=native -std=c11 '
    '-DREQUANT_MODE=1 '
    '-DDATA_HEADER=`"qmnist_data.h`" '
    '-DTEST_IMAGES_SYM=test_images '
    '-DTEST_LABELS_SYM=test_labels '
    '-DNTEST=10000 '
    'qcnn.c export_correct.c -o export_correct'
)

print(f"[info] ヘッダ出力先: {OUT_H}")
print("[info] PowerShell での実行例（場所: src_c2 フォルダ）:\n")
print("PS " + str(SRC_DIR) + "> " + ps_one_liner)


[ok] Wrote header: D:\cfs\final\src_c2\qmnist_data.h  (NTEST default=10000)
[info] ヘッダ出力先: D:\cfs\final\src_c2\qmnist_data.h
[info] PowerShell での実行例（場所: src_c2 フォルダ）:

PS D:\cfs\final\src_c2> gcc -O3 -march=native -std=c11 -DREQUANT_MODE=1 -DDATA_HEADER=`"qmnist_data.h`" -DTEST_IMAGES_SYM=test_images -DTEST_LABELS_SYM=test_labels -DNTEST=10000 qcnn.c export_correct.c -o export_correct
